

# Word2Vec Tutorial

Obs.: cópia adaptada (corrigida) de https://medium.com/@manansuri/a-dummys-guide-to-word2vec-456444f3c673

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the Dataset + Text Preprocessing

In [ ]:
dataset = pd.read_csv('tweets.csv', encoding='latin1')

In [ ]:
dataset

Very basic text preproccessing by removing punctuations, numbers. We have also cleaned the data by removing characters including and after 'https' in the text.

In [ ]:
import re
texts=[]
for i in range(0,len(dataset)):
  text = re.sub('[^a-zA-Z]', ' ', str(dataset['text'][i]))
  text = text.lower()
  text = text.split()
  x = len(text) if text.count('https') ==0  else text.index('https')
  text = text[: x ]
  text = [t for t in text if not t=='https']
  text = ' '.join(text)
  texts.append(text)


Size of the dataset.

In [ ]:
print(len(texts))

In [ ]:
sentences = [line.split() for line in texts]
print(sentences[20:24])

## Training the word2vec model

In [ ]:
! pip install gensim scikit-learn

In [ ]:
from gensim.models import Word2Vec


Running the Word2Vec model

In [ ]:

# sentences	- List of tokenized sentences	[['i','like','cats']]
# vector_size	- Embedding dimension	100	Word vector length
# window	- Context window size
# epochs	- Training passes over training data
# min_count	- Min frequency to include a word
# sg	0 = CBOW, 1 = Skip-gram	0	Model type
w2v = Word2Vec(sentences, vector_size=300, window=200, epochs=200, min_count=1, sg=0)

In [ ]:
print(sentences[20:24])

## Working with word2vec

Finding the vocabulary of the model can be useful in several general applications, and in this case, provides us with a list of words we can try and use other functions.

In [ ]:
# words = list(w2v.wv.vocab)
words = w2v.wv.index_to_key

print(words)

In [ ]:
print(len(words))

Finding the embedding of a given word can be useful when we’re trying to represent sentences as a collection of word embeddings, like when we’re trying to make a weight matrix for the embedding layer of a network.

In [ ]:
print(w2v.wv['rolo'] )

We can also find out the similarity between given words (the cosine distance between their vectors).

In [ ]:
w2v.wv.similarity('cin', 'recife')

With the gensim, we can also find the most similar words to a given word.

In [ ]:
print(w2v.wv.most_similar('caba'))

In [ ]:
print(w2v.wv.most_similar('cac'))

## Visualising word vectors

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def display_pca_scatterplot(model, words=None, sample=0):
    if words == None:
        if sample > 0:
            words = np.random.choice(list(model.wv.key_to_index.keys()), sample)
        else:
            words = [ word for word in model.vocab ]

    # word_vectors = np.array([model[w] for w in words])
    word_vectors = np.array([model.wv[w] for w in words])

    twodim = PCA().fit_transform(word_vectors)[:,:2]

    plt.figure(figsize=(6,6))
    plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r')
    for word, (x,y) in zip(words, twodim):
        plt.text(x+0.05, y+0.05, word)

In [ ]:
display_pca_scatterplot(w2v,words)

In [ ]:
display_pca_scatterplot(w2v, ['palavras', 'para', 'plotar'])